# Week 8 Capstone — Meeting-to-Action Assistant

Self-contained capstone notebook. It reuses Week 3 structured prompting, Week 4 validation, Week 5 MiniLM/FAISS retrieval, Week 6 cross-encoder reranking, and Week 7 dynamic few-shot selection.

**Architecture:** transcript → retrieval → reranking → optional examples → Gemini Flash → validation → evaluation.

The held-out set contains 10 meeting examples. Evaluation reports deterministic extraction metrics plus Gemini-as-judge scores for **correctness**, **grounding**, and **operational usefulness**. A required failure is traced with the A/B/C ablation and a mitigation is measured. The research note is generated dynamically at the end from the executed results.

In [ ]:
# Colab, once if needed:
# %pip install -q sentence-transformers faiss-cpu openai pandas numpy tabulate

import os,re,json,time,math,hashlib
import numpy as np,pandas as pd,faiss
from sentence_transformers import SentenceTransformer,CrossEncoder
from IPython.display import display,Markdown

try:
    from google.colab import userdata
    if not os.getenv("GEMINI_API_KEY"):
        os.environ["GEMINI_API_KEY"]=userdata.get("GEMINI_API_KEY")
except Exception:
    pass

GEMINI_MODEL=os.getenv("GEMINI_MODEL","gemini-3.5-flash")
JUDGE_MODEL=os.getenv("GEMINI_JUDGE_MODEL",GEMINI_MODEL)
CASES=[["E01", "A: Maybe delay launch. B: No, keep Friday. B: I'll send QA by 3.", "keep friday", "send qa", "B", "", ""], ["E02", "A: The export is missing IDs. B: I'll ask Finance to fix them.", "", "ask finance", "B", "missing ids", ""], ["E03", "A: Could we use vendor B? B: Legal has not reviewed it. A: Do not choose yet.", "do not choose", "", "", "legal not reviewed", ""], ["E04", "A: Weekly dashboard might help. B: No cadence decision. B: I'll collect examples.", "", "collect examples", "B", "", ""], ["E05", "A: Import failed on invalid dates. B: I'll fix dates today. A: Keep current mapping.", "keep current mapping", "fix dates", "B", "invalid dates", ""], ["E06", "A: Use Sep 1 through Sep 30 for both reports. B: I'll rerun them tomorrow.", "use sep 1 through sep 30", "rerun reports", "B", "", ""], ["E07", "A: We could automate email. B: Not in v1; human review is required. A: I'll document the gate.", "human review is required", "document the gate", "A", "", ""], ["E08", "A: Duplicate customers cause bad totals. B: I'll test customer ID as the key.", "", "test customer id", "B", "duplicate customers", ""], ["E09", "A: Should we remove notes? B: Not yet. We agreed to keep the shorter form. A: I'll measure completion time.", "keep the shorter form", "measure completion time", "A", "", "remove notes"], ["E10", "A: SSO effort is unknown. B: Do not promise it yet. B: I'll ask Engineering.", "do not promise sso", "ask engineering", "B", "sso effort unknown", ""]]
RULES=[
    "Decision requires explicit commitment; suggestions are not decisions.",
    "Action item is future work; never invent an owner.",
    "Blocker materially delays or risks progress.",
    "Follow-up question must remain unresolved."
]
SHOTS=[
    ("Maybe delay. No, keep Friday.","decision: keep Friday"),
    ("I will ask Finance.","action_item"),
    ("Legal has not reviewed it.","blocker"),
    ("Should we remove notes?","follow_up_question")
]
emb=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
rr=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
rv=emb.encode(RULES,normalize_embeddings=True).astype("float32"); ri=faiss.IndexFlatIP(rv.shape[1]); ri.add(rv)
sv=emb.encode([x[0] for x in SHOTS],normalize_embeddings=True).astype("float32"); si=faiss.IndexFlatIP(sv.shape[1]); si.add(sv)
CACHE={"synthesis":{},"judge":{}}

def client():
    from openai import OpenAI
    return OpenAI(api_key=os.environ["GEMINI_API_KEY"],base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

def parse(s):
    s=re.sub(r"^```(?:json)?\s*|\s*```$","",s.strip(),flags=re.I)
    return json.loads(s)

def ret(q,docs,index,text=lambda x:x,k=3):
    v=emb.encode([q],normalize_embeddings=True).astype("float32")
    _,ids=index.search(v,min(6,len(docs)))
    cand=[docs[i] for i in ids[0] if i>=0]
    scores=rr.predict([(q,text(x)) for x in cand])
    return [x for _,x in sorted(zip(scores,cand),reverse=True)[:k]]

def key(x):
    return hashlib.sha256(json.dumps(x,sort_keys=True).encode()).hexdigest()

def extract(t,cfg):
    ck=key([GEMINI_MODEL,cfg,t])
    if ck in CACHE["synthesis"]:
        return CACHE["synthesis"][ck]
    prompt='Return JSON only with summary, decisions, actions, blockers, questions. Decisions must be confirmed; do not invent owners; unresolved questions only.'
    if cfg in "BC":
        prompt+="\nGuidance:\n"+"\n".join(ret(t,RULES,ri))
    if cfg=="C":
        prompt+="\nExamples:\n"+"\n".join(a+" => "+b for a,b in ret(t,SHOTS,si,lambda x:x[0]))
    st=time.perf_counter()
    r=client().chat.completions.create(model=GEMINI_MODEL,messages=[{"role":"user","content":prompt+"\nTranscript:\n"+t}],temperature=0)
    try:
        brief=parse(r.choices[0].message.content)
    except Exception:
        brief={}
    out={"brief":brief,"valid":all(x in brief for x in ["summary","decisions","actions","blockers","questions"]),"latency":time.perf_counter()-st}
    CACHE["synthesis"][ck]=out
    return out

STOP=set("the a an to of and or for in on by with is are be as at this that".split())
def tok(s): return set(re.findall(r"[a-z0-9]+",str(s).lower()))-STOP
def sim(a,b):
    A,B=tok(a),tok(b)
    return len(A&B)/len(A|B) if A and B else 0
def hit(pred,gold):
    return 1 if gold and any(sim(x,gold)>=.25 for x in pred) else (1 if not gold and not pred else 0)

def score(c,r):
    _,t,gd,ga,go,gb,gq=c
    b=r["brief"] if r["valid"] else {}
    d=[x.get("text","") if isinstance(x,dict) else str(x) for x in b.get("decisions",[])]
    a=[x.get("text","") if isinstance(x,dict) else str(x) for x in b.get("actions",[])]
    bl=[x.get("text","") if isinstance(x,dict) else str(x) for x in b.get("blockers",[])]
    q=[x.get("text","") if isinstance(x,dict) else str(x) for x in b.get("questions",[])]
    owner=1 if not ga else int(any(isinstance(x,dict) and x.get("owner","").lower()==go.lower() and sim(x.get("text",""),ga)>=.25 for x in b.get("actions",[])))
    return {"case":c[0],"decision_f1":hit(d,gd),"action_f1":hit(a,ga),"owner_accuracy":owner,"blocker_f1":hit(bl,gb),"question_f1":hit(q,gq),"valid":int(r["valid"]),"latency_s":r["latency"]}

## A/B/C ablation

A = transcript only; B = retrieved/reranked guidance; C = guidance + dynamic few-shot examples.

In [ ]:
RESULTS={}; rows=[]
if not os.getenv("GEMINI_API_KEY"):
    print("API-BLOCKED: add GEMINI_API_KEY.")
else:
    for cfg in "ABC":
        RESULTS[cfg]={}
        for c in CASES:
            r=extract(c[1],cfg); RESULTS[cfg][c[0]]=r
            x=score(c,r); x["config"]=cfg; rows.append(x)
    METRICS=pd.DataFrame(rows)
    SUMMARY=METRICS.groupby("config").mean(numeric_only=True).round(3)
    display(METRICS); display(SUMMARY)

## LLM-as-judge

All 10 held-out cases are judged for each configuration on correctness, grounding, and operational usefulness.

In [ ]:
JUDGE_DF=pd.DataFrame(); JUDGE_SUMMARY=pd.DataFrame()
def judge(c,b):
    payload={"transcript":c[1],"gold":c[2:],"candidate":b}
    ck=key([JUDGE_MODEL,payload])
    if ck in CACHE["judge"]: return CACHE["judge"][ck]
    prompt='Score 1-5 on correctness, grounding, operational_usefulness. Return JSON only with those keys plus reason.'
    r=client().chat.completions.create(model=JUDGE_MODEL,messages=[{"role":"user","content":prompt+"\n"+json.dumps(payload)}],temperature=0)
    out=parse(r.choices[0].message.content); CACHE["judge"][ck]=out; return out
if RESULTS:
    jr=[]
    for cfg in "ABC":
        for c in CASES:
            jr.append({"config":cfg,"case":c[0],**judge(c,RESULTS[cfg][c[0]]["brief"])})
    JUDGE_DF=pd.DataFrame(jr)
    JUDGE_SUMMARY=JUDGE_DF.groupby("config")[["correctness","grounding","operational_usefulness"]].mean().round(3)
    display(JUDGE_DF); display(JUDGE_SUMMARY)

## Required failure mode and mitigation

In [ ]:
FAILURE=None; MITIGATION_BEFORE=MITIGATION_AFTER=None
if RESULTS:
    sc={cfg:{c[0]:score(c,RESULTS[cfg][c[0]]) for c in CASES} for cfg in "ABC"}
    chosen=comp=None
    for c in CASES:
        if sc["C"][c[0]]["decision_f1"]<sc["B"][c[0]]["decision_f1"]:
            chosen,comp=c[0],"dynamic few-shot context"; break
    if chosen is None:
        for c in CASES:
            if sc["B"][c[0]]["decision_f1"]<sc["A"][c[0]]["decision_f1"]:
                chosen,comp=c[0],"retrieved/reranked context"; break
    if chosen is None:
        chosen=min((c[0] for c in CASES),key=lambda x:sc["C"][x]["decision_f1"]); comp="generation/extraction stage"
    FAILURE=(chosen,comp)
    print("Failure case:",chosen,"| component:",comp)

    HEDGE=["maybe","could","might","consider","possibly"]; COMMIT=["agreed","approved","keep","use","do not","required"]
    def mitigate(b):
        z=json.loads(json.dumps(b))
        out=[]
        for d in z.get("decisions",[]):
            e=d.get("evidence","").lower() if isinstance(d,dict) else str(d).lower()
            if any(h in e for h in HEDGE) and not any(k in e for k in COMMIT): continue
            out.append(d)
        z["decisions"]=out
        return z

    before=[]; after=[]
    for c in CASES:
        r=RESULTS["C"][c[0]]; before.append(score(c,r)["decision_f1"])
        q=dict(r); q["brief"]=mitigate(r["brief"]); after.append(score(c,q)["decision_f1"])
    MITIGATION_BEFORE=round(np.mean(before),3); MITIGATION_AFTER=round(np.mean(after),3)
    print("Decision F1 before:",MITIGATION_BEFORE,"after:",MITIGATION_AFTER)

## Research note

Generated dynamically from the executed results below.

In [ ]:
def research_note():
    if not RESULTS or JUDGE_DF.empty or FAILURE is None:
        display(Markdown("### Measured results\n\nComplete the evaluation above first.")); return None
    cols=["decision_f1","action_f1","owner_accuracy","blocker_f1","question_f1","valid","latency_s"]
    det=METRICS.groupby("config")[cols].mean().round(3)
    jud=JUDGE_SUMMARY
    scorev={c:float(det.loc[c,["decision_f1","action_f1","owner_accuracy"]].mean()) for c in "ABC"}
    best=min([c for c,v in scorev.items() if v==max(scorev.values())],key=lambda x:"ABC".index(x))
    retrieval="improved" if scorev["B"]>scorev["A"] else ("regressed" if scorev["B"]<scorev["A"] else "did not change")
    few="improved" if scorev["C"]>scorev["B"] else ("regressed" if scorev["C"]<scorev["B"] else "did not change")
    delta=MITIGATION_AFTER-MITIGATION_BEFORE
    note=f'''### Measured results

#### A/B/C deterministic metric table

{det.to_markdown()}

#### LLM-as-judge averages

{jud.to_markdown()}

#### Failure mode
Case **{FAILURE[0]}** was traced to **{FAILURE[1]}**.

#### Mitigation
Decision F1 changed from **{MITIGATION_BEFORE:.3f}** to **{MITIGATION_AFTER:.3f}** ({delta:+.3f}).

#### Final architecture decision
Use **Configuration {best}**. The selection score is the mean of decision F1, action-item F1, and owner accuracy; ties prefer the simpler configuration.

#### Answers
- **Did retrieval improve the system?** It {retrieval} the measured extraction score from A to B.
- **Did dynamic few-shot prompting help?** It {few} the measured extraction score from B to C.
- **What failed?** Case **{FAILURE[0]}**, traced to **{FAILURE[1]}**.
- **Did mitigation help?** Decision F1 changed by **{delta:+.3f}**.
- **What is the final architecture?** **Configuration {best}**, selected from the measured results rather than complexity alone.

No result here is hard-coded; this note is generated from the executed measurements above.
'''
    display(Markdown(note)); return note
RESEARCH_NOTE=research_note()

## Submission guard

In [ ]:
problems=[]
if not RESULTS: problems.append("A/B/C evaluation missing")
if JUDGE_DF.empty or len(JUDGE_DF)!=30: problems.append("judge pass missing/incomplete")
if FAILURE is None: problems.append("failure trace missing")
if RESEARCH_NOTE is None: problems.append("research note missing")
print("READY FOR SUBMISSION" if not problems else "NOT READY FOR SUBMISSION")
for p in problems: print("-",p)
print("\nCACHE_EXPORT =",json.dumps(CACHE,separators=(",",":")))
print("\nHTML export: jupyter nbconvert --to html week8_meeting_to_action_capstone.ipynb")